# **SYNTHETIC DATASET GENERATION**
### NOTE: Aggregate statistics are taken from the original data to replicate the results maintaining anonymity 

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta 

## **Static columns: Age, Gender, Type of cancer, Type of ICI treatment, AKI date, AKI flag, End of follow-up date, Death flag**

In [2]:
# Fix random seed for reproducibility 
np.random.seed(42)

In [3]:
# Function to generate a synthetic dataset containing n patients 
def dataset_generator(n):

    dataset = []
    
    # Age (initialization)
    age = np.zeros(n, dtype=int)
    
    # Gender 
    gender = np.random.choice(['M', 'F'], size=n, p=[0.6, 0.4])
    
    # Type of cancer (common categories treated with ICIs)
    cancer_types = ['Melanoma', 'Lung', 'Urogenital', 'Other']
    cancer = np.random.choice(cancer_types, size=n, p=[0.1, 0.3, 0.2, 0.4])

    # Type of ICI 
    ici_types = ['CTLA-4', 'PD-1', 'PD-L1', 'CTLA-4 + PD-1/PD-L1', 'Other']
    ici = np.random.choice(ici_types, size=n, p=[0.01, 0.5, 0.3, 0.15, 0.04])

    # AKI flag
    aki_flag = np.random.choice([0, 1], size=n, p=[0.85, 0.15])

    # Date generation for time-to-event analysis (2019)
    base_date = pd.to_datetime('2019-01-01')
    
    for i in range(n):

        # ID 
        patient_id = f'PT_{i:03d}'

        # ICI start date (between 2019-2021)
        start_date = base_date + timedelta(days=np.random.randint(0, 1300))

        # AKI date (typically around 10-150 days)
        aki_date = pd.NaT

        # Max follow-up for all the patients (7 years)
        max_followup_days = np.random.randint(1830, 2555)

        # Increase death probability if type of cancer is 'Other' (rare malignancies) or the type of ICI is 'Other' (ICIs under clinical trial)
        modifier = 1.0 
    
        if ici[i] == 'Other':
            modifier *= 1.15

        if cancer[i] == 'Other':
            modifier *= 1.15
        
        if aki_flag[i] == 1:
            # Median time 90 days 
            days_to_aki = int(np.random.lognormal(mean=np.log(90), sigma=0.5))
            aki_date = start_date + timedelta(days=max(1, days_to_aki))
            
            # Age (greater for AKI patients)
            age[i] = int(np.random.normal(loc=66, scale=9.6))
            
            # Higher probability of death for AKI patients (0.9)
            prob_death_aki = min(1.0, 0.9 * modifier) 
            death_flag = np.random.choice([0, 1], p=[1 - prob_death_aki, prob_death_aki])

            if death_flag == 1:

                # If the patient with AKI dies, there are two possible subgroups (Early vs Late Death)
                is_early_death = np.random.choice([0, 1], p = [0.5, 0.5])

                if is_early_death == 1:
                    # Median time to death: 14 days  
                    days_to_death = np.random.lognormal(mean=np.log(14), sigma=0.4)
                    days_to_death = np.clip(days_to_death, 1, 90)

                else:
                    days_to_death = np.random.randint(91, max_followup_days)
                
                # Final date (death or censoring)
                final_date = aki_date + timedelta(days=int(days_to_death))
            
            else:
                final_date = aki_date + timedelta(days=np.random.randint(200, 1825))
            
        else:
            # Age (lower for No AKI patients)
            age[i] = int(np.random.normal(loc=63, scale=11.8))
            
            # Lower probability of death for No AKI patients (0.8) - still high due to long-term follow-up  
            prob_death_no_aki = min(1.0, 0.8 * modifier) 
            death_flag = np.random.choice([0, 1], p = [1 - prob_death_no_aki, prob_death_no_aki])

            if death_flag == 1:                
                # Final date (death or censoring)
                final_date = start_date + timedelta(days=np.random.randint(30, max_followup_days))
            else:
                final_date = start_date + timedelta(days=max_followup_days)
            
        dataset.append({'ID': patient_id, 'ICI_start': start_date, 'age': age[i], 'gender': gender[i], 'cancer_type': cancer[i], 'ICI_type': ici[i], 'AKI': aki_flag[i], 'AKI_date': aki_date, 'death_flag': death_flag, 'final_date': final_date})
    
    # DataFrame creation 
    df = pd.DataFrame(dataset)
        
    # Ensure the age is in a realistic range 
    df['age'] = df['age'].clip(30, 90)
    
    return df

# Number of patients 
n = 752
data = dataset_generator(n)

In [4]:
data.head(10)

,ID,ICI_start,age,gender,cancer_type,ICI_type,AKI,AKI_date,death_flag,final_date
0,PT_000,2020-04-08,48,M,Other,PD-1,0,NaT,1,2022-10-26
1,PT_001,2022-01-25,57,F,Lung,PD-1,0,NaT,1,2028-01-29
2,PT_002,2021-06-06,55,F,Lung,PD-L1,0,NaT,1,2023-07-01
3,PT_003,2019-07-16,60,M,Other,PD-L1,0,NaT,1,2020-11-08
4,PT_004,2019-10-16,71,M,Other,PD-L1,0,NaT,1,2020-06-16
5,PT_005,2019-05-08,73,M,Urogenital,PD-L1,0,NaT,0,2025-08-30
6,PT_006,2021-11-18,63,M,Other,CTLA-4 + PD-1/PD-L1,1,2022-02-14,1,2026-01-23
7,PT_007,2020-11-11,55,F,Other,PD-L1,0,NaT,1,2024-04-21
8,PT_008,2021-10-15,83,F,Lung,CTLA-4 + PD-1/PD-L1,0,NaT,1,2026-11-14
9,PT_009,2019-07-27,74,F,Lung,PD-1,0,NaT,1,2025-01-08


## **Longitudinal columns: serum creatinine (sCr)**
### **sCr is simulated using percentage changes with respect to baseline, replicating the original data**

In [5]:
def append_longitudinal_creatinine(df_base):
    
    time_stamps = [3, 12, 24, 36, 60]
    
    creatinine_data = []
    
    for _, row in df_base.iterrows():
        patient_id = row['ID']
        start_date = row['ICI_start']
        final_date = row['final_date']
        gender = row['gender']
        has_aki = row['AKI']
        aki_date = row['AKI_date']

        # 1. Baseline sCr (t_0: start of ICI therapy): men tend to have greater sCr due to greater muscle mass
        if gender == 'M':
            base_scr = np.random.lognormal(mean=np.log(0.82), sigma=0.15)
        else:
            base_scr = np.random.lognormal(mean=np.log(0.70), sigma=0.15)

        # Baaseline sCr is an AKI predictor 
        if has_aki == 1:
            base_scr *= 1.15

        base_scr = np.clip(base_scr, 0.4, 2.5)
        
       # Dictionary for each patient 
        patient_record = {'ID': patient_id, 'cr_0m': round(base_scr, 2)}

        # 2. Longitudinal monitoring 
        for t in time_stamps:
            visit_date = start_date + pd.DateOffset(months=t)

            # Clinical drop-out: NaN if the visit happens after the end of follow-up or death 
            if pd.notna(final_date) and visit_date > final_date:
                patient_record[f'cr_{t}m'] = np.nan
                continue

            # Add probability of missing data at random (MAR): 10% (missing visits)
            if np.random.rand() < 0.1:
                patient_record[f'cr_{t}m'] = np.nan 
                continue 
            
            # sCr variation modelling 
            if has_aki == 0:
                # No AKI: stable sCr up to 12 months, then gradual increase to 10% at 60m
                if t <= 12:
                    perc_change = np.random.normal(loc=0.01, scale=0.05)
                else:
                    # Linear scaling of the percentage change between 12m and 60m
                    expected_change = 0.10 * ((t - 12) / 48)
                    perc_change = np.random.normal(loc=expected_change, scale=0.08)
            else:
                # AKI: increase of 20% and to 40% at 60 months 
                if pd.notna(aki_date) and visit_date >= aki_date: 
                    if t <= 36:
                        perc_change = np.random.normal(loc=0.20, scale=0.15)
                    else:
                        perc_change = np.random.normal(loc=0.40, scale=0.20)

                else:
                    # If AKI did not happen
                    perc_change = np.random.normal(loc=0.01, scale=0.05)

            # Apply percentage change 
            current_scr = base_scr * (1 + perc_change)

            # Limit to a physiological range 
            current_scr = np.clip(current_scr, 0.4, 12.0)
          
            patient_record[f'cr_{t}m'] = round(current_scr, 2)
            
        creatinine_data.append(patient_record)
        
    df_cr = pd.DataFrame(creatinine_data)
    
    # Merge using ID as key
    df_final = pd.merge(df_base, df_cr, on='ID', how='left')
    
    return df_final

# Function call
complete_data = append_longitudinal_creatinine(data)

## **Final example**

In [6]:
complete_data.sample(20)

,ID,ICI_start,age,gender,cancer_type,ICI_type,AKI,AKI_date,death_flag,final_date,cr_0m,cr_3m,cr_12m,cr_24m,cr_36m,cr_60m
318,PT_318,2020-07-04,64,M,Lung,PD-L1,0,NaT,0,2026-04-20,0.65,0.63,0.77,0.68,0.64,NaN
386,PT_386,2022-05-27,63,M,Urogenital,PD-1,0,NaT,1,2024-10-20,0.77,0.71,0.77,0.83,NaN,NaN
540,PT_540,2021-10-12,82,F,Urogenital,CTLA-4 + PD-1/PD-L1,0,NaT,1,2025-03-21,0.86,NaN,0.86,0.87,0.83,NaN
430,PT_430,2020-09-28,64,M,Other,PD-L1,0,NaT,1,2023-08-19,0.77,0.78,0.82,0.78,NaN,NaN
658,PT_658,2020-03-08,80,F,Lung,PD-1,0,NaT,0,2026-07-20,0.84,0.85,0.77,NaN,0.81,0.98
511,PT_511,2019-05-02,72,M,Urogenital,PD-1,0,NaT,1,2022-11-27,0.83,0.84,0.86,NaN,0.97,NaN
44,PT_044,2021-04-04,68,M,Other,PD-1,0,NaT,1,2022-09-10,1.17,NaN,1.14,NaN,NaN,NaN
242,PT_242,2021-07-28,79,F,Lung,PD-1,1,2021-08-24,1,2021-08-29,0.73,NaN,NaN,NaN,NaN,NaN
460,PT_460,2020-06-07,52,M,Lung,PD-L1,0,NaT,1,2025-08-18,0.98,0.97,NaN,1.02,1.12,NaN
248,PT_248,2021-05-26,59,F,Lung,PD-1,0,NaT,1,2026-05-05,0.89,0.91,0.90,0.87,1.02,NaN


In [7]:
# Save the data 
complete_data.to_csv('synthetic_dataset.csv', index=False)